# 05 — Event Studies & Empirical Contagion Persistence Analysis

### Academic Research Objectives:
1. **Empirical Case Studies:** Analyze fine-grained hourly network propagation dynamics across documented historical crises:
   - **May 2022:** Terra/LUNA algorithmic stablecoin collapse.
   - **November 2022:** FTX / Alameda systemic bankruptcy.
   - **August 2024:** Global Yen carry-trade unwind flash crash.
2. **Persistence Quantification:** Measure propagation delay, time-to-hit (hours), affected receiver nodes, and decay half-life.
3. **Persistence by Shock Type & Sector:** Non-parametric tests (Kruskal-Wallis) comparing contagion persistence across volatility shocks, liquidity squeezes, and regulatory/exchange shocks.


In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import torch
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import kruskal

from cryptograph.analysis.event_study import EventStudyAnalyzer


## 1. Load Hourly Crisis Windows from External Events Catalogue


In [ ]:
pers_file = Path('../reports/persistence_analysis.csv')
if pers_file.exists():
    df_pers = pd.read_csv(pers_file)
    print('=== Historical Crisis Shock Propagation & Persistence ===')
    print(df_pers[['event_name', 'timestamp', 'shock_type', 'blast_radius_count', 'blast_radius_pct', 'propagation_delay_hours', 'mean_persistence_half_life_hours', 'most_impacted_sector']].to_string())
else:
    print('reports/persistence_analysis.csv not found.')


## 2. Case Study 1: The August 2024 Yen Carry Unwind Flash Crash


In [ ]:
if pers_file.exists():
    yen_event = df_pers[df_pers['event_name'].str.contains('Yen', case=False, na=False)]
    if len(yen_event) > 0:
        row = yen_event.iloc[0]
        print('Case Study: August 2024 Yen Carry-Trade Unwind')
        print(f'- Initial Transmitter Coins: {row["initial_transmitters"]}')
        print(f'- Blast Radius: {row["blast_radius_count"]} coins ({row["blast_radius_pct"]*100:.1f}% of universe)')
        print(f'- Propagation Delay: {row["propagation_delay_hours"]} hours to maximum cross-asset transmission')
        print(f'- Volatility Persistence Half-Life: {row["mean_persistence_half_life_hours"]:.1f} hours')
        print(f'- Primary Sector Impacted: {row["most_impacted_sector"]}')


## 3. Case Study 2: The November 2022 FTX Collapse


In [ ]:
events_df = pd.read_csv('../data/external/events.csv')
print('=== Historical Events Catalog ===')
print(events_df[['date', 'name', 'is_crisis']].to_string())


## 4. Cross-Sector Persistence & Kruskal-Wallis Non-Parametric Significance


In [ ]:
from cryptograph.analysis.persistence import ShockPersistenceAnalyzer
analyzer = ShockPersistenceAnalyzer()
kw_res = analyzer.compute_sector_kruskal_wallis(
    dataset_path='../data/processed/v2/1h/processed_dataset.pt',
    sectors_csv_path='../data/external/sectors.csv'
)
print('=== Kruskal-Wallis Non-Parametric Test Across Sectors ===')
print(f'- H-Statistic: {kw_res["kruskal_statistic"]:.4f}')
print(f'- p-value: {kw_res["p_value"]:.4e}')
print(f'- Evaluated Sectors: {kw_res["num_sectors"]}')
if kw_res["p_value"] < 0.05:
    print('=> Statistically significant: Shock persistence durations differ fundamentally across sectors (p < 0.05).')
else:
    print('=> Null hypothesis not rejected.')
